# Exploring the Allmaps APIs from a IIIF manifest

[Allmaps](https://allmaps.org/) is an open source project that lets people georeference (align to real-world coordinates) scanned historical maps that are published through [IIIF](https://iiif.io/) (International Image Interoperability Framework). This notebook starts from nothing but a IIIF manifest URL and walks through the Allmaps APIs step by step, showing how to:

1. Ask Allmaps whether it already has a georeference annotation for a manifest.
2. Pull the Allmaps Map ID out of that annotation.
3. Build the URLs for the Allmaps Editor (to georeference a map), Allmaps Viewer (to view a georeferenced map), and Allmaps XYZ tile server (to use the map as a tile layer, e.g. in Leaflet or MapLibre).
4. Use the Allmaps REST API to look up map and manifest metadata directly.

Each code cell below is preceded by a short explanation, and every line inside the code is commented for readers who are new to Python and/or to spatial data APIs.


## Step 1: Ask Allmaps for a Georeference Annotation

Given nothing but a IIIF manifest URL, we can ask Allmaps' hosted [annotations API](https://github.com/allmaps/allmaps/tree/main/apis/annotations) whether it already has a Georeference Annotation for that manifest. The annotations API accepts any IIIF resource URL (manifest, canvas, or image) as a `url` query parameter and returns an `AnnotationPage` containing zero or more Georeference Annotations.


In [ ]:
import requests  # requests lets Python make HTTP calls to web APIs, like a browser without the UI
import json  # kept for reference; requests already parses JSON responses into Python objects for us

# define manifest
# hard coded for this example

manifest = "https://purl.stanford.edu/tr202jn9182/iiif/manifest"  # the IIIF Presentation manifest URL we start from

# print the georef annotation for that manifest as JSON data
# pass the manifest as a params value so requests URL-encodes it correctly

allmapsAPIRequest = requests.get("https://annotations.allmaps.org/", params={"url": manifest})
# requests.get() sends an HTTP GET request; params={"url": manifest} appends "?url=<percent-encoded manifest>" for us

if allmapsAPIRequest.status_code >= 500:  # a 5xx status means the server itself failed, not just "not found"
    # A 500 here usually means Allmaps could not fetch/parse the manifest itself.
    # Check directly whether the manifest URL is reachable by a plain HTTP client.
    manifestCheck = requests.get(manifest)  # try fetching the manifest ourselves, to compare against Allmaps' failure
    if "text/html" in manifestCheck.headers.get("Content-Type", ""):  # HTML instead of JSON usually means a bot/login wall
        raise RuntimeError(
            "The manifest URL returned an HTML page instead of IIIF JSON "
            "(likely a bot/reCAPTCHA verification gate), so Allmaps could not "
            "fetch or parse it either. Use a manifest URL that is reachable "
            "without a browser session, e.g. the LMEC example later in this notebook."
        )

allmapsAPIRequest.raise_for_status()  # raise an exception for any remaining 4xx/5xx error, so failures aren't silent
georefAnnotation = allmapsAPIRequest.json()  # parse the HTTP response body from JSON text into a Python dict


print(georefAnnotation)  # show the raw AnnotationPage dict so we can see its structure


{'id': 'https://annotations.allmaps.org/manifests/cd78c03407a95d51', 'type': 'AnnotationPage', '@context': 'http://www.w3.org/ns/anno.jsonld', 'items': [{'id': 'https://annotations.allmaps.org/maps/a4c21a56d19cf41c', 'type': 'Annotation', '@context': ['http://iiif.io/api/extension/georef/1/context.json', 'http://iiif.io/api/presentation/3/context.json'], 'created': '2026-08-24T17:27:48.857Z', 'modified': '2026-08-24T17:27:48.857Z', 'motivation': 'georeferencing', 'target': {'type': 'SpecificResource', 'source': {'id': 'https://stacks.stanford.edu/image/iiif/tr202jn9182%2FAM_0228', 'type': 'ImageService2', 'height': 8638, 'width': 10261, 'partOf': [{'id': 'https://purl.stanford.edu/tr202jn9182/iiif/canvas/cocina-fileSet-tr202jn9182-c23a15fa-ab6e-4906-b112-bf410971384e', 'type': 'Canvas', 'label': {'none': ['Image 1']}, 'partOf': [{'id': 'https://purl.stanford.edu/tr202jn9182/iiif/manifest', 'type': 'Manifest', 'label': {'none': ['South Africa, from official & other authentic authorities

## Step 2: Extract the Allmaps Map ID

Every Georeference Annotation Allmaps hosts has an `id` field shaped like `https://annotations.allmaps.org/maps/{mapId}`. The last path segment is the **Allmaps Map ID**, a short identifier used across all of the other Allmaps APIs and apps. See the [`@allmaps/annotation` package](https://github.com/allmaps/allmaps/tree/main/packages/annotation) for the annotation shape, and the [annotations API](https://github.com/allmaps/allmaps/tree/main/apis/annotations) for how `items` is returned.


In [ ]:
# detect Allmaps Map ID in georef anno

mapID = None  # start with no map ID; we'll fill this in once we find one in the response

# handle both successful payload shapes and error payloads
items = georefAnnotation.get("item") or georefAnnotation.get("items")
# .get(...) looks up a dict key without raising an error if it's missing (returns None instead)
# some Allmaps responses use "item" (singular) and some use "items" (plural), so we check both

if isinstance(items, dict):  # isinstance() checks the Python type; a single annotation may come back as one dict, not a list
    items = [items]  # wrap it in a list so the rest of this cell can always assume a list

if items:  # a non-empty list is "truthy" in Python, so this checks "did we get anything back?"
    raw_id = items[0].get("id", "")  # take the first annotation's "id" field, defaulting to "" if missing
    mapID = raw_id.rsplit("/", 1)[-1]  # split the URL on the last "/" and keep only the final piece (the map ID)
else:
    raise ValueError(f"Could not find map ID. API response: {georefAnnotation}")  # fail loudly if nothing was found

print(mapID)  # show the extracted Allmaps Map ID, e.g. "a4c21a56d19cf41c"

a4c21a56d19cf41c


## Step 3: Rebuild the Georeference Annotation URL from the Map ID

Once you have a Map ID, you can always get back to its Georeference Annotation at `https://annotations.allmaps.org/maps/{mapId}`, documented in the [Allmaps annotations API](https://github.com/allmaps/allmaps/tree/main/apis/annotations).


In [ ]:
# construct georef annotation URL from map ID

annoBaseUrl = "https://annotations.allmaps.org/maps/"  # the annotations API's base path for looking up a single map
georefAnnoUrl = annoBaseUrl+mapID  # string concatenation: base URL + map ID = a full, valid annotation URL

print(georefAnnoUrl)  # e.g. https://annotations.allmaps.org/maps/a4c21a56d19cf41c

https://annotations.allmaps.org/maps/a4c21a56d19cf41c


## Step 4: Recap what we have so far

We now have three related identifiers for the same georeferenced map: the original IIIF manifest, its Allmaps Map ID, and its Georeference Annotation URL. These three pieces are the building blocks for every other Allmaps app and API endpoint documented in the [allmaps/allmaps](https://github.com/allmaps/allmaps) monorepo.


In [ ]:
# now we have a georef anno endpoint + map ID just from the IIIF manifest

print("IIIF Manifest URL: "+manifest+"\n\r")  # the string we started with
print("Map ID: "+mapID+"\n\r")  # the short identifier we extracted in Step 2
print("Georeference Annotation URL: "+georefAnnoUrl+"\n\r")  # the annotation URL we rebuilt in Step 3


IIIF Manifest URL: https://purl.stanford.edu/tr202jn9182/iiif/manifest

Map ID: a4c21a56d19cf41c

Georeference Annotation URL: https://annotations.allmaps.org/maps/a4c21a56d19cf41c



## Step 5: Build an Allmaps Editor link

[Allmaps Editor](https://github.com/allmaps/allmaps/tree/main/apps/editor) is the web app people use to georeference IIIF maps by hand: placing control points on the scanned image and matching them to real-world coordinates. It accepts a `url` query parameter pointing at any IIIF manifest, image, or collection, and opens straight to that resource so a new georeferencing session can begin.


In [ ]:
# construct Editor endpoint with URL parameter

editorBaseUrl = "https://editor.allmaps.org/#/collection?url="  # Editor's hash-based route for opening a IIIF resource
editorEndpoint = editorBaseUrl+manifest  # append our manifest URL so Editor opens directly to it

print(editorEndpoint)  # paste this URL into a browser to start georeferencing

https://editor.allmaps.org/#/collection?url=https://purl.stanford.edu/tr202jn9182/iiif/manifest


## Step 6 (optional): Add a callback URL to Editor

Some institutions integrate directly with [Allmaps Editor](https://github.com/allmaps/allmaps/tree/main/apps/editor) using an additional `callback` parameter, which Editor can use to send the user back to the source catalog record once georeferencing is complete. This only works for institutions Allmaps has explicitly configured for callbacks, which is why this example switches to a Leventhal Map & Education Center (LMEC) manifest instead of the Stanford one used above.


In [ ]:
# optionally add a callback URL to the header of Editor
# only works with certain institutions currently
# which is why example uses LMEC map

lmecObject = "https://collections.leventhalmap.org/search/commonwealth:0z709604j"  # the LMEC catalog page for this map
lmecManifest = lmecObject+"/manifest"  # LMEC's convention: appending "/manifest" gives the IIIF manifest URL
allmapsCallbackURL = "https://editor.allmaps.org/#/collection?url=" + lmecManifest + "&callback="+ lmecObject;
# builds one URL with two query parameters: "url" (what to georeference) and "callback" (where to return afterward)

print(allmapsCallbackURL)


https://editor.allmaps.org/#/collection?url=https://collections.leventhalmap.org/search/commonwealth:0z709604j/manifest&callback=https://collections.leventhalmap.org/search/commonwealth:0z709604j


## Step 7: Build an Allmaps Viewer link (search endpoint)

[Allmaps Viewer](https://github.com/allmaps/allmaps/tree/main/apps/viewer) renders an already-georeferenced map on a real-world basemap. Its `url` query parameter expects something that resolves to a Georeference Annotation. Here we point it at the [annotations search endpoint](https://github.com/allmaps/allmaps/tree/main/apis/annotations) (`annotations.allmaps.org/?url=...`), which is a good sanity check but not the canonical form Viewer expects — see Step 8 for the direct annotation URL.


In [ ]:
# construct Viewer endpoint with URL parameter

georefAnnotationUrlParam = "https://annotations.allmaps.org/?url="+manifest
# this nests the annotations search endpoint (?url=manifest) inside Viewer's own url parameter
viewerBaseUrl = "https://viewer.allmaps.org/?url="  # Viewer's own query parameter for "what annotation to show"
viewerEndpointUrlParam = viewerBaseUrl+georefAnnotationUrlParam  # combine the two into one nested URL

print(viewerEndpointUrlParam)

https://viewer.allmaps.org/?url=https://annotations.allmaps.org/?url=https://purl.stanford.edu/tr202jn9182/iiif/manifest


## Step 8: Build an Allmaps Viewer link (direct annotation)

This is the canonical way to open [Allmaps Viewer](https://github.com/allmaps/allmaps/tree/main/apps/viewer): point its `url` parameter directly at a single Georeference Annotation URL (`annotations.allmaps.org/maps/{mapId}`), the same one built in Step 3.


In [ ]:
# construct Viewer endpoint with pure Georeference Annotation

viewerEndpointPureAnno = viewerBaseUrl+georefAnnoUrl  # reuse Viewer's base URL, but with a single-annotation URL this time

print(viewerEndpointPureAnno)  # this is the URL pattern documented for Allmaps Viewer

https://viewer.allmaps.org/?url=https://annotations.allmaps.org/maps/a4c21a56d19cf41c


## Step 9: Build an XYZ tile URL from an annotation URL

[Allmaps Tile Server](https://github.com/allmaps/allmaps/blob/main/workers/tileserver/README.md) (`allmaps.xyz`) turns any Georeference Annotation into standard [XYZ map tiles](https://en.wikipedia.org/wiki/Tiled_web_map), the same tile format used by Leaflet, MapLibre, and most other web mapping libraries. One way to supply the annotation is via a `url` query parameter pointing at any reachable Georeference Annotation URL.


In [ ]:
# construct XYZ tile endpoint from URL param

xyzBaseUrl = "https://allmaps.xyz/{z}/{x}/{y}.png?url="
# {z}/{x}/{y} are placeholders a map library fills in with zoom/column/row when it requests a tile
xyzEndpointUrlParam = xyzBaseUrl+georefAnnoUrl  # point the tile server at our Georeference Annotation URL

print(xyzEndpointUrlParam)

https://allmaps.xyz/{z}/{x}/{y}.png?url=https://annotations.allmaps.org/maps/a4c21a56d19cf41c


## Step 10: Build an XYZ tile URL from a Map ID (shorter form)

[Allmaps Tile Server](https://github.com/allmaps/allmaps/blob/main/workers/tileserver/README.md) also supports a shorter `maps/{mapId}/{z}/{x}/{y}.png` route when you already know the Allmaps Map ID, avoiding an extra lookup.


In [ ]:
# construct XYZ tile endpoint from map ID

print(f"https://allmaps.xyz/maps/{mapID}/{{z}}/{{x}}/{{y}}.png")
# an f-string substitutes mapID directly; the doubled {{ }} are needed to print literal { } characters for z/x/y

https://allmaps.xyz/maps/a4c21a56d19cf41c/{z}/{x}/{y}.png


## Step 11: Look up metadata with the Allmaps REST API

[api.allmaps.org](https://github.com/allmaps/allmaps/tree/main/apis/rest) is Allmaps' REST API for map, image, canvas, and manifest metadata (ground control points, resource masks, transformation type, and more). Note that the manifest ID is not available from a `/maps/{id}/manifests` sub-route — it's nested inside the map's own `_allmaps` metadata block, so we fetch the map first and then read the manifest ID out of it.


In [ ]:
# use Allmaps API to parse annotations using Map IDs

apiEndpoint = f'https://api.allmaps.org/maps/{mapID}'  # REST endpoint for this map's full metadata

# the manifest ID is embedded in the map's own _allmaps metadata, there is no /manifests sub-route on /maps/{id}
mapInfo = requests.get(apiEndpoint).json()  # fetch the map metadata and parse it as a Python dict
manifestID = mapInfo["_allmaps"]["image"]["canvases"][0]["manifests"][0]["id"].rsplit("/", 1)[-1]
# dig through the nested dict/list structure to find the manifest's own id, then keep only its last path segment

print("Get point, polygon, and metadata for given Map ID: "+apiEndpoint+"\n\r")
print("Get Manifest metadata for given Manifest ID: https://api.allmaps.org/manifests/"+manifestID)


Get point, polygon, and metadata for given Map ID: https://api.allmaps.org/maps/a4c21a56d19cf41c

Get Manifest metadata for given Manifest ID: https://api.allmaps.org/manifests/cd78c03407a95d51
